# 2.0 — Pydantic Models

Exercises the schemas in `src/data/schemas/`: valid construction, derived
properties, and validation failures. The final section enriches via PubChem
(network) and is optional.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent  # the notebook lives in notebooks/ -> repo root
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

%load_ext autoreload
%autoreload 2

from datetime import date

from src.data.schemas import ATCCode, Drug, Interaction, Med, Patient, SideEffect

## Valid construction

A polymedicated patient with one medication (amoxicillin) and its active
principle.

In [ ]:
atc = ATCCode(code="J01CA04")  # name resolved from codes.json

rash = SideEffect(
    name="rash",
    description="skin rash",
    severity="mild",
)

interaction = Interaction(
    interacting_drug="warfarin",
    mechanism="increased anticoagulant effect",
    management="monitor INR",
)

amox = Drug(
    cid=33613,
    name="amoxicillin",
    chemical_group=atc,
    molecular_formula="C16H19N3O5S",
    side_effects=[rash],
    interactions=[interaction],
)

med = Med(
    ATC_code=atc,
    name="Amoxicillin 500mg",
    dosage="500 mg",
    frequency="every 8 hours",
    start_date=date(2026, 6, 1),
    end_date=date(2026, 6, 10),
    active_principles=[amox],
)

patient = Patient(
    id=1,
    name="Test patient",
    age=78,
    birth_date=date(1948, 1, 1),
    number_of_meds=1,
    polymedicated=True,
    diseases=["respiratory infection"],
    medication=[med],
)

patient.model_dump()

## Derived properties

In [ ]:
print("med.is_active :", med.is_active)   # end_date in the past -> False
print("med.duration  :", med.duration, "days")
print("atc.level     :", atc.level)
print("atc.class     :", atc.pharmacological_class)

## Validation failures

Each cell triggers an error and shows its message. Pydantic's
`ValidationError` wraps the `ValueError` raised by each validator.

In [ ]:
from pydantic import ValidationError

# 1) ATC code with invalid format -> validation.invalid_atc_code
try:
    ATCCode(code="ZZZ99")
except ValidationError as exc:
    print(exc)

In [ ]:
# 2) Two active principles with the same CID -> unique_active_principles
try:
    Med(
        ATC_code=atc,
        name="dup",
        dosage="1",
        frequency="1/d",
        start_date=date(2026, 6, 1),
        active_principles=[
            Drug(cid=33613, name="amoxicillin"),
            Drug(cid=33613, name="amoxicillin (dup)"),
        ],
    )
except ValidationError as exc:
    print(exc)

In [ ]:
# 3) start_date after end_date -> check_dates
try:
    Med(
        ATC_code=atc,
        name="dates",
        dosage="1",
        frequency="1/d",
        start_date=date(2026, 6, 10),
        end_date=date(2026, 6, 1),
    )
except ValidationError as exc:
    print(exc)

In [ ]:
# 4) age inconsistent with birth_date -> check_age_matches_birth_date
try:
    Patient(
        id=2,
        name="bad age",
        age=40,
        birth_date=date(1948, 1, 1),  # ~78 real years
        number_of_meds=0,
        polymedicated=False,
        diseases=[],
        medication=[],
    )
except ValidationError as exc:
    print(exc)

## Enrichment via PubChem — OPTIONAL ⚠️

`get_drug_by_cid` and `build_med` query the **PubChem API** (network). They
are not required to validate the notebook. CID 33613 = amoxicillin.

In [ ]:
from src.data.repository import build_med, get_drug_by_cid

drug = get_drug_by_cid(33613)  # resolves name/formula/SMILES/InChIKey from PubChem
print(drug.model_dump() if drug else "not resolved")

enriched_med = build_med(
     atc_code="J01CA04",
     name="Amoxicillin 500mg",
     dosage="500 mg",
     frequency="every 8 hours",
     start_date=date(2026, 6, 1),
     cids=[33613],
)
enriched_med.model_dump()